In [ ]:
%pip install ecdsa
%pip install bit

In [ ]:
# Generate valid Casascius-style mini keys from index 1..10000 and print:
#   <minikey> <p2pkh_compressed> <p2pkh_uncompressed> <privkey_hex>
#
# Hardcoded: start=1, end=10000, minikey length=22 ('S' + 21 base58 chars)
# Validity: SHA256(minikey + "?")[0] == 0x00

import hashlib

try:
    import ecdsa  # pip install ecdsa
except ImportError as e:
    raise SystemExit("Missing dependency: ecdsa. Install with: pip install ecdsa") from e

# ---------- Base58 (for addresses) ----------
B58_ALPHABET = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"

def b58encode(b: bytes) -> str:
    n = int.from_bytes(b, "big")
    zeros = len(b) - len(b.lstrip(b"\x00"))
    out = []
    while n > 0:
        n, rem = divmod(n, 58)
        out.append(B58_ALPHABET[rem])
    out_str = "".join(reversed(out)) if out else "1"
    return ("1" * zeros) + out_str

def b58check_encode(payload: bytes) -> str:
    chk = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    return b58encode(payload + chk)

# ---------- RIPEMD-160 (with fallback to pycryptodome if needed) ----------
def ripemd160(data: bytes) -> bytes:
    try:
        h = hashlib.new("ripemd160")
        h.update(data)
        return h.digest()
    except ValueError:
        try:
            from Crypto.Hash import RIPEMD
            return RIPEMD.new(data).digest()
        except Exception as e:
            raise SystemExit(
                "RIPEMD-160 not available. Install pycryptodome: pip install pycryptodome"
            ) from e

# ---------- Mini key generation from index ----------
def index_to_minikey(index: int, total_len: int = 22) -> str:
    if index < 1:
        raise ValueError("index must be >= 1")
    body_len = total_len - 1
    n = index - 1  # index 1 -> all '1's
    digits = [0] * body_len
    i = body_len - 1
    while n > 0 and i >= 0:
        digits[i] = n % 58
        n //= 58
        i -= 1
    body = "".join(B58_ALPHABET[d] for d in digits)
    return "S" + body

def is_valid_minikey(minikey: str) -> bool:
    h = hashlib.sha256((minikey + "?").encode("ascii")).digest()
    return h[0] == 0x00

# ---------- Keys & addresses ----------
def privkey_from_minikey(minikey: str) -> bytes:
    return hashlib.sha256(minikey.encode("ascii")).digest()

def pubkey_from_priv(priv32: bytes, compressed: bool) -> bytes:
    sk = ecdsa.SigningKey.from_string(priv32, curve=ecdsa.SECP256k1)
    vk = sk.verifying_key
    x = vk.pubkey.point.x()
    y = vk.pubkey.point.y()
    x_bytes = x.to_bytes(32, "big")
    y_bytes = y.to_bytes(32, "big")
    if compressed:
        prefix = 0x02 if (y % 2 == 0) else 0x03
        return bytes([prefix]) + x_bytes
    else:
        return b"\x04" + x_bytes + y_bytes

def p2pkh_address_from_pubkey(pubkey_bytes: bytes, version: int = 0x00) -> str:
    h160 = ripemd160(hashlib.sha256(pubkey_bytes).digest())
    payload = bytes([version]) + h160
    return b58check_encode(payload)

# ---------- Main loop (hardcoded range) ----------
START = 1
END = 10_00
MINI_LEN = 22  # 'S' + 21 base58 chars

def main():
    for idx in range(START, END + 1):
        mk = index_to_minikey(idx, MINI_LEN)
        if not is_valid_minikey(mk):
            continue
        priv = privkey_from_minikey(mk)
        priv_hex = priv.hex()
        pub_c = pubkey_from_priv(priv, compressed=True)
        pub_u = pubkey_from_priv(priv, compressed=False)
        addr_c = p2pkh_address_from_pubkey(pub_c, version=0x00)  # mainnet
        addr_u = p2pkh_address_from_pubkey(pub_u, version=0x00)
        print(f"{mk} {addr_c} {addr_u} {priv_hex}")

if __name__ == "__main__":
    main()


In [ ]:
WifSolverCuda.exe -miniStartIndex 1 -miniEndIndex 10000 -afile address.txt -b 1 -t 32 -s 64

In [ ]:
from bit import Key

In [ ]:
Key.from_hex("A127E55C8F2886F9154EE3CA983F994975740B46AD6A5F1EDD59C49F7E0F50F1").address

In [ ]:

C:\Users\vrgaw\Documents\WifSolverCuda\x64\Debug>WifSolverCuda.exe -miniStartIndex 1 -miniEndIndex 1000 -afile address.txt -b 368 -t 256 -c
WifSolver 0.6.2

Use parameter '-h' for help and list of available parameters

S111111111111111111111
S1111111111111111111JE
Using GPU nr 0:
NVIDIA GeForce RTX 4070 (46 procs)
maxThreadsPerBlock: 1024

Mini-key index mode
  start mini-key: S111111111111111111111
  end   mini-key: S1111111111111111111JE
  blocks=368 threads=256 steps=5000

Work started at Sun Sep 14 19:55:49 2025

number of blocks: 368
number of threads: 256
number of checks per thread: 5000

[b43 t77] tid=11085 tIx=55425000 mini=S11111111111111115u4vT
[b27 t68] tid=6980 tIx=34900000 mini=S111111111111111145sZ9
[b1 t134] tid=390 tIx=1950000 mini=S11111111111111111Azfh
[b28 t238] tid=7406 tIx=37030000 mini=S11111111111111114GnjH
[b31 t18] tid=7954 tIx=39770000 mini=S11111111111111114WqEf
[b5 t153] tid=1433 tIx=7165000 mini=S11111111111111111diuV
[b7 t218] tid=2010 tIx=10050000 mini=S11111111111111111tWWs
[b17 t0] tid=4352 tIx=21760000 mini=S11111111111111112vXVR
[b1 t251] tid=507 tIx=2535000 mini=S11111111111111111DzZu
[b34 t33] tid=8737 tIx=43685000 mini=S11111111111111114ru2f
[b31 t207] tid=8143 tIx=40715000 mini=S11111111111111114bg9m
[b26 t207] tid=6863 tIx=34315000 mini=S111111111111111142sew
[b23 t108] tid=5996 tIx=29980000 mini=S11111111111111113ef1Z
[b39 t150] tid=10134 tIx=50670000 mini=S11111111111111115UhRh
[b6 t137] tid=1673 tIx=8365000 mini=S11111111111111111jsd9
[b21 t82] tid=5458 tIx=27290000 mini=S11111111111111113QsNF
[b16 t86] tid=4182 tIx=20910000 mini=S11111111111111112rApF
[b16 t139] tid=4235 tIx=21175000 mini=S11111111111111112sXbD
[b6 t120] tid=1656 tIx=8280000 mini=S11111111111111111jSMd
[b36 t195] tid=9411 tIx=47055000 mini=S11111111111111115AAp7
[b21 t235] tid=5611 tIx=28055000 mini=S11111111111111113Unmu
[b21 t247] tid=5623 tIx=28115000 mini=S11111111111111113V6cP
[b34 t116] tid=8820 tIx=44100000 mini=S11111111111111114u2Pq
[b16 t191] tid=4287 tIx=21435000 mini=S11111111111111112trsy
[b32 t59] tid=8251 tIx=41255000 mini=S11111111111111114eSg7
[b21 t27] tid=5403 tIx=27015000 mini=S11111111111111113PTcs
[b9 t176] tid=2480 tIx=12400000 mini=S111111111111111126Z67
[b40 t125] tid=10365 tIx=51825000 mini=S11111111111111115acmV
[b32 t211] tid=8403 tIx=42015000 mini=S11111111111111114iLbZ
[b36 t161] tid=9377 tIx=46885000 mini=S111111111111111159JH5
[b45 t150] tid=11670 tIx=58350000 mini=S11111111111111116A4RV
[b45 t154] tid=11674 tIx=58370000 mini=S11111111111111116AANK
[b29 t124] tid=7548 tIx=37740000 mini=S11111111111111114LRnf
[b21 t144] tid=5520 tIx=27600000 mini=S11111111111111113STX5
[b21 t159] tid=5535 tIx=27675000 mini=S11111111111111113SqpB
[b14 t165] tid=3749 tIx=18745000 mini=S11111111111111112f5Ef
[b14 t245] tid=3829 tIx=19145000 mini=S11111111111111112h89D
[b30 t218] tid=7898 tIx=39490000 mini=S11111111111111114VQ15
[b24 t165] tid=6309 tIx=31545000 mini=S11111111111111113ngEK
[b9 t15] tid=2319 tIx=11595000 mini=S111111111111111122Rno
[b9 t24] tid=2328 tIx=11640000 mini=S111111111111111122fAf
[b40 t225] tid=10465 tIx=52325000 mini=S11111111111111115dBQB
[b25 t181] tid=6581 tIx=32905000 mini=S11111111111111113ueWb
[b2 t215] tid=727 tIx=3635000 mini=S11111111111111111KdZR
[b2 t216] tid=728 tIx=3640000 mini=S11111111111111111Kf3d
[b23 t27] tid=5915 tIx=29575000 mini=S11111111111111113caco
[b20 t82] tid=5202 tIx=26010000 mini=S11111111111111113JJsJ
[b20 t156] tid=5276 tIx=26380000 mini=S11111111111111113LCrc
[b20 t196] tid=5316 tIx=26580000 mini=S11111111111111113MEJt
[b27 t244] tid=7156 tIx=35780000 mini=S11111111111111114AP9a
[b8 t219] tid=2267 tIx=11335000 mini=S1111111111111111216W4
[b4 t224] tid=1248 tIx=6240000 mini=S11111111111111111YywE
[b0 t35] tid=35 tIx=175000 mini=S111111111111111111u2G
[b16 t56] tid=4152 tIx=20760000 mini=S11111111111111112qQE4
[b32 t237] tid=8429 tIx=42145000 mini=S11111111111111114j1Ex
[b32 t104] tid=8296 tIx=41480000 mini=S11111111111111114fbZS
[b7 t143] tid=1935 tIx=9675000 mini=S11111111111111111rb3N
[b21 t166] tid=5542 tIx=27710000 mini=S11111111111111113T2De
[b7 t230] tid=2022 tIx=10110000 mini=S11111111111111111tpMN
[b30 t229] tid=7909 tIx=39545000 mini=S11111111111111114VgMN
[b30 t241] tid=7921 tIx=39605000 mini=S11111111111111114VzBr
[b30 t142] tid=7822 tIx=39110000 mini=S11111111111111114TT3N
[b32 t144] tid=8336 tIx=41680000 mini=S11111111111111114gd1i
[b1 t112] tid=368 tIx=1840000 mini=S11111111111111111ARyA
[b42 t218] tid=10970 tIx=54850000 mini=S11111111111111115r7zg
[b9 t228] tid=2532 tIx=12660000 mini=S111111111111111127tNt
[b17 t247] tid=4599 tIx=22995000 mini=S111111111111111132rcY
[b17 t250] tid=4602 tIx=23010000 mini=S111111111111111132w5A
[b17 t121] tid=4473 tIx=22365000 mini=S11111111111111112ydLU
[b19 t21] tid=4885 tIx=24425000 mini=S11111111111111113ABhi
[b28 t141] tid=7309 tIx=36545000 mini=S11111111111111114EJZE
[b28 t211] tid=7379 tIx=36895000 mini=S11111111111111114G6bi
[b28 t177] tid=7345 tIx=36725000 mini=S11111111111111114FE4g
[b19 t238] tid=5102 tIx=25510000 mini=S11111111111111113FkEc
[b19 t255] tid=5119 tIx=25595000 mini=S11111111111111113GBW8
[b6 t187] tid=1723 tIx=8615000 mini=S11111111111111111m9wW
[b5 t7] tid=1287 tIx=6435000 mini=S11111111111111111ZyuJ
[b5 t15] tid=1295 tIx=6475000 mini=S11111111111111111aBnx
[b38 t59] tid=9787 tIx=48935000 mini=S11111111111111115Kofv
[b34 t11] tid=8715 tIx=43575000 mini=S11111111111111114rLL8
[b34 t23] tid=8727 tIx=43635000 mini=S11111111111111114reAc
[b15 t42] tid=3882 tIx=19410000 mini=S11111111111111112iUvC
[b15 t158] tid=3998 tIx=19990000 mini=S11111111111111112mTLC
[b15 t167] tid=4007 tIx=20035000 mini=S11111111111111112mgi4
[b15 t218] tid=4058 tIx=20290000 mini=S11111111111111112nzWc
[b3 t157] tid=925 tIx=4625000 mini=S11111111111111111QhrQ
[b41 t78] tid=10574 tIx=52870000 mini=S11111111111111115fyQk
[b28 t81] tid=7249 tIx=36245000 mini=S11111111111111114CmNq
[b8 t34] tid=2082 tIx=10410000 mini=S11111111111111111vMXo
[b20 t240] tid=5360 tIx=26800000 mini=S11111111111111113NMi1
[b39 t233] tid=10217 tIx=51085000 mini=S11111111111111115Wpnu
[b9 t115] tid=2419 tIx=12095000 mini=S111111111111111124zRX
[b31 t249] tid=8185 tIx=40925000 mini=S11111111111111114ckaV
[b8 t235] tid=2283 tIx=11415000 mini=S111111111111111121WHP
[b44 t10] tid=11274 tIx=56370000 mini=S11111111111111115yuqb
[b44 t131] tid=11395 tIx=56975000 mini=S1111111111111111631gd
[b9 t77] tid=2381 tIx=11905000 mini=S1111111111111111241wf
[b17 t78] tid=4430 tIx=22150000 mini=S11111111111111112xXRb
[b4 t149] tid=1173 tIx=5865000 mini=S11111111111111111X4Tj
[b4 t208] tid=1232 tIx=6160000 mini=S11111111111111111Ya9w
[b35 t33] tid=8993 tIx=44965000 mini=S11111111111111114yTXf
[b44 t62] tid=11326 tIx=56630000 mini=S111111111111111161F8M
[b23 t182] tid=6070 tIx=30350000 mini=S11111111111111113gYzu
[b12 t20] tid=3092 tIx=15460000 mini=S11111111111111112NEim
[b23 t255] tid=6143 tIx=30715000 mini=S11111111111111113iRW1
[b24 t6] tid=6150 tIx=30750000 mini=S11111111111111113ibuT
[b13 t247] tid=3575 tIx=17875000 mini=S11111111111111112acch
[b14 t148] tid=3732 tIx=18660000 mini=S11111111111111112edyB
[b30 t58] tid=7738 tIx=38690000 mini=S11111111111111114RJC1
[b42 t4] tid=10756 tIx=53780000 mini=S11111111111111115kdvR
[b19 t131] tid=4995 tIx=24975000 mini=S11111111111111113D1CV
[b7 t161] tid=1953 tIx=9765000 mini=S11111111111111111s3o7
[b35 t187] tid=9147 tIx=45735000 mini=S111111111111111153QRX
[b11 t121] tid=2937 tIx=14685000 mini=S11111111111111112JGLh
[b42 t76] tid=10828 tIx=54140000 mini=S11111111111111115nUwK
[b42 t119] tid=10871 tIx=54355000 mini=S11111111111111115oarD
[b25 t63] tid=6463 tIx=32315000 mini=S11111111111111113rd8D
[b2 t8] tid=520 tIx=2600000 mini=S11111111111111111EKtd
[b3 t70] tid=838 tIx=4190000 mini=S11111111111111111NUYR
[b25 t14] tid=6414 tIx=32070000 mini=S11111111111111113qNJ5
[b22 t10] tid=5642 tIx=28210000 mini=S11111111111111113VarM
[b29 t140] tid=7564 tIx=37820000 mini=S11111111111111114Lqa1
[b29 t166] tid=7590 tIx=37950000 mini=S11111111111111114MWDP
[b18 t38] tid=4646 tIx=23230000 mini=S1111111111111111344UH
[b25 t151] tid=6551 tIx=32755000 mini=S11111111111111113tsvR
[b37 t224] tid=9696 tIx=48480000 mini=S11111111111111115HUR7
[b37 t232] tid=9704 tIx=48520000 mini=S11111111111111115HgJm
[b6 t63] tid=1599 tIx=7995000 mini=S11111111111111111hyds
[b16 t206] tid=4302 tIx=21510000 mini=S11111111111111112uFB8
[b20 t177] tid=5297 tIx=26485000 mini=S11111111111111113Lk4z
[b16 t241] tid=4337 tIx=21685000 mini=S11111111111111112v9CN
[b23 t71] tid=5959 tIx=29795000 mini=S11111111111111113di1x
[b43 t109] tid=11117 tIx=55585000 mini=S11111111111111115utV8
[b17 t132] tid=4484 tIx=22420000 mini=S11111111111111112yugn
[b12 t108] tid=3180 tIx=15900000 mini=S11111111111111112QVWz
[b12 t49] tid=3121 tIx=15605000 mini=S11111111111111112Nypn
[b12 t214] tid=3286 tIx=16430000 mini=S11111111111111112TD4v
[b17 t166] tid=4518 tIx=22590000 mini=S11111111111111112znDp
[b40 t20] tid=10260 tIx=51300000 mini=S11111111111111115Xvhp
[b0 t73] tid=73 tIx=365000 mini=S111111111111111112sWA
[b35 t196] tid=9156 tIx=45780000 mini=S111111111111111153doQ
[b35 t212] tid=9172 tIx=45860000 mini=S1111111111111111543ai
[b27 t216] tid=7128 tIx=35640000 mini=S111111111111111149fXp
[b24 t196] tid=6340 tIx=31700000 mini=S11111111111111113oUJn
[b1 t1] tid=257 tIx=1285000 mini=S111111111111111117azE
[b24 t91] tid=6235 tIx=31175000 mini=S11111111111111113knF4
[b30 t6] tid=7686 tIx=38430000 mini=S11111111111111114PxuG
[b15 t102] tid=3942 tIx=19710000 mini=S11111111111111112k26e
[b26 t154] tid=6810 tIx=34050000 mini=S111111111111111141Wt2
[b7 t56] tid=1848 tIx=9240000 mini=S11111111111111111pMjQ
[b38 t153] tid=9881 tIx=49405000 mini=S11111111111111115NDPQ
[b33 t45] tid=8493 tIx=42465000 mini=S11111111111111114keNE
[b6 t82] tid=1618 tIx=8090000 mini=S11111111111111111iTsp
[b37 t223] tid=9695 tIx=48475000 mini=S11111111111111115HSvv
[b14 t104] tid=3688 tIx=18440000 mini=S11111111111111112dWa6
[b14 t43] tid=3627 tIx=18135000 mini=S11111111111111112bwuU
[b18 t170] tid=4778 tIx=23890000 mini=S111111111111111137Sfc
[b21 t107] tid=5483 tIx=27415000 mini=S11111111111111113RWXV
[b32 t67] tid=8259 tIx=41295000 mini=S11111111111111114eeZq
[b32 t11] tid=8203 tIx=41015000 mini=S11111111111111114dDLF
[b9 t216] tid=2520 tIx=12600000 mini=S111111111111111127aYT
[b5 t45] tid=1325 tIx=6625000 mini=S11111111111111111axPD
[b13 t119] tid=3447 tIx=17235000 mini=S11111111111111112XLNF
[b43 t16] tid=11024 tIx=55120000 mini=S11111111111111115sWFu
[b30 t106] tid=7786 tIx=38930000 mini=S11111111111111114SXXy
[b30 t113] tid=7793 tIx=38965000 mini=S11111111111111114ShwR
[b30 t117] tid=7797 tIx=38985000 mini=S11111111111111114SotF
[b34 t151] tid=8855 tIx=44275000 mini=S11111111111111114uvR9
[b26 t74] tid=6730 tIx=33650000 mini=S11111111111111113yTyV
[b11 t18] tid=2834 tIx=14170000 mini=S11111111111111112FdFR
[b3 t221] tid=989 tIx=4945000 mini=S11111111111111111SLyh
[b10 t166] tid=2726 tIx=13630000 mini=S11111111111111112Crj5
[b10 t57] tid=2617 tIx=13085000 mini=S11111111111111112A4iX
[b45 t180] tid=11700 tIx=58500000 mini=S11111111111111116Aq1m
[b45 t49] tid=11569 tIx=57845000 mini=S111111111111111167UJf
[b33 t147] tid=8595 tIx=42975000 mini=S11111111111111114oFyM
[b41 t165] tid=10661 tIx=53305000 mini=S11111111111111115iCio
[b24 t249] tid=6393 tIx=31965000 mini=S11111111111111113pq5n
[b24 t145] tid=6289 tIx=31445000 mini=S11111111111111113nAWG
[b44 t237] tid=11501 tIx=57505000 mini=S111111111111111165jEc
[b20 t24] tid=5144 tIx=25720000 mini=S11111111111111113GpfN
[b31 t186] tid=8122 tIx=40610000 mini=S11111111111111114b8wW
[b4 t13] tid=1037 tIx=5185000 mini=S11111111111111111TaKe
[b31 t93] tid=8029 tIx=40145000 mini=S11111111111111114YkiG
[b31 t113] tid=8049 tIx=40245000 mini=S11111111111111114ZGSQ
[b27 t7] tid=6919 tIx=34595000 mini=S111111111111111144Jtc
[b30 t171] tid=7851 tIx=39255000 mini=S11111111111111114UC9S
[b35 t86] tid=9046 tIx=45230000 mini=S11111111111111114zpJg
[b35 t91] tid=9051 tIx=45255000 mini=S11111111111111114zwji
[b42 t147] tid=10899 tIx=54495000 mini=S11111111111111115pJU4
[b34 t212] tid=8916 tIx=44580000 mini=S11111111111111114wV5n
[b19 t79] tid=4943 tIx=24715000 mini=S11111111111111113Bfun
[b38 t21] tid=9749 tIx=48745000 mini=S11111111111111115JqC8
[b11 t194] tid=3010 tIx=15050000 mini=S11111111111111112L8qr
[b11 t61] tid=2877 tIx=14385000 mini=S11111111111111112GjAL
[b3 t115] tid=883 tIx=4415000 mini=S11111111111111111PdRn
[b19 t205] tid=5069 tIx=25345000 mini=S11111111111111113EuBr
[b26 t251] tid=6907 tIx=34535000 mini=S111111111111111144148
[b45 t64] tid=11584 tIx=57920000 mini=S111111111111111167rbn
[b22 t204] tid=5836 tIx=29180000 mini=S11111111111111113aZCY
[b37 t116] tid=9588 tIx=47940000 mini=S11111111111111115Ehtp
[b18 t122] tid=4730 tIx=23650000 mini=S111111111111111136DKi
[b36 t51] tid=9267 tIx=46335000 mini=S111111111111111156UnR
[b13 t80] tid=3408 tIx=17040000 mini=S11111111111111112WLQD
[b13 t14] tid=3342 tIx=16710000 mini=S11111111111111112UeJZ
[b13 t139] tid=3467 tIx=17335000 mini=S11111111111111112Xr6R
[b13 t153] tid=3481 tIx=17405000 mini=S11111111111111112YCuK
[b4 t58] tid=1082 tIx=5410000 mini=S11111111111111111UjCy
[b12 t179] tid=3251 tIx=16255000 mini=S11111111111111112SK3j
[b35 t123] tid=9083 tIx=45415000 mini=S111111111111111151mJM
[b39 t179] tid=10163 tIx=50815000 mini=S11111111111111115VSXo
[b0 t128] tid=128 tIx=640000 mini=S111111111111111114HFb
[b42 t241] tid=10993 tIx=54965000 mini=S11111111111111115riBX
[b19 t54] tid=4918 tIx=24590000 mini=S11111111111111113B2kd
[b11 t139] tid=2955 tIx=14775000 mini=S11111111111111112Jj6V
[b14 t199] tid=3783 tIx=18915000 mini=S11111111111111112fwmo
[b37 t40] tid=9512 tIx=47560000 mini=S11111111111111115Ckw7
[b41 t133] tid=10629 tIx=53145000 mini=S11111111111111115hPAD
[b10 t102] tid=2662 tIx=13310000 mini=S11111111111111112BDbs
[b37 t79] tid=9551 tIx=47755000 mini=S11111111111111115DkuB
[b22 t244] tid=5876 tIx=29380000 mini=S11111111111111113baeq
[b41 t219] tid=10715 tIx=53575000 mini=S11111111111111115jaz1
[b16 t16] tid=4112 tIx=20560000 mini=S11111111111111112pNmt
[b27 t151] tid=7063 tIx=35315000 mini=S111111111111111147zvS
[b43 t168] tid=11176 tIx=55880000 mini=S11111111111111115wQBQ
[b0 t101] tid=101 tIx=505000 mini=S111111111111111113b82
[b42 t160] tid=10912 tIx=54560000 mini=S11111111111111115pdnn
[b42 t35] tid=10787 tIx=53935000 mini=S11111111111111115mRzv
[b30 t76] tid=7756 tIx=38780000 mini=S11111111111111114Rkwp
[b15 t13] tid=3853 tIx=19265000 mini=S11111111111111112hjpJ
[b15 t250] tid=4090 tIx=20450000 mini=S11111111111111112op5L
[b26 t36] tid=6692 tIx=33460000 mini=S11111111111111113xVVg
[b26 t96] tid=6752 tIx=33760000 mini=S11111111111111113z2g6
[b38 t234] tid=9962 tIx=49810000 mini=S11111111111111115QHnE
[b38 t107] tid=9835 tIx=49175000 mini=S11111111111111115M31x
[b45 t235] tid=11755 tIx=58775000 mini=S11111111111111116CEmC
[b29 t71] tid=7495 tIx=37475000 mini=S11111111111111114K51p
[b2 t179] tid=691 tIx=3455000 mini=S11111111111111111Ji46
[b2 t111] tid=623 tIx=3115000 mini=S11111111111111111Gxz2
[b2 t65] tid=577 tIx=2885000 mini=S11111111111111111FncW
[b18 t225] tid=4833 tIx=24165000 mini=S111111111111111138rR4
[b29 t231] tid=7655 tIx=38275000 mini=S11111111111111114PApv
[b2 t157] tid=669 tIx=3345000 mini=S11111111111111111J9MY
[b20 t101] tid=5221 tIx=26105000 mini=S11111111111111113Jo7M
[b31 t130] tid=8066 tIx=40330000 mini=S11111111111111114Zhhy
[b23 t201] tid=6089 tIx=30445000 mini=S11111111111111113h3Ew
[b21 t223] tid=5599 tIx=27995000 mini=S11111111111111113UUwZ
[b1 t82] tid=338 tIx=1690000 mini=S111111111111111119fP5
[b8 t115] tid=2163 tIx=10815000 mini=S11111111111111111xRvf
[b40 t177] tid=10417 tIx=52085000 mini=S11111111111111115bx4P
[b36 t93] tid=9309 tIx=46545000 mini=S111111111111111157ZD9
[b3 t54] tid=822 tIx=4110000 mini=S11111111111111111N4mD
[b2 t230] tid=742 tIx=3710000 mini=S11111111111111111L1rf
[b2 t44] tid=556 tIx=2780000 mini=S11111111111111111FFQB
[b41 t242] tid=10738 tIx=53690000 mini=S11111111111111115kBAo
[b41 t243] tid=10739 tIx=53695000 mini=S11111111111111115kCf1
[b18 t24] tid=4632 tIx=23160000 mini=S111111111111111133hfV
[b33 t244] tid=8692 tIx=43460000 mini=S11111111111111114qk9V
[b22 t189] tid=5821 tIx=29105000 mini=S11111111111111113aAuV
[b10 t68] tid=2628 tIx=13140000 mini=S11111111111111112AM4s
[b10 t79] tid=2639 tIx=13195000 mini=S11111111111111112AdR9
[b33 t24] tid=8472 tIx=42360000 mini=S11111111111111114k79y
[b28 t98] tid=7266 tIx=36330000 mini=S11111111111111114DCeU
[b9 t148] tid=2452 tIx=12260000 mini=S111111111111111125qUU
[b43 t218] tid=11226 tIx=56130000 mini=S11111111111111115xgVn
[b43 t61] tid=11069 tIx=55345000 mini=S11111111111111115tf9J
[b44 t199] tid=11463 tIx=57315000 mini=S111111111111111164kkp
[b44 t208] tid=11472 tIx=57360000 mini=S111111111111111164z8g
[b0 t190] tid=190 tIx=950000 mini=S111111111111111115sQU
[b36 t234] tid=9450 tIx=47250000 mini=S11111111111111115BAnL
[b15 t77] tid=3917 tIx=19585000 mini=S11111111111111112jNwa
[b34 t236] tid=8940 tIx=44700000 mini=S11111111111111114x6kp
[b3 t9] tid=777 tIx=3885000 mini=S11111111111111111Lusv
[b33 t203] tid=8651 tIx=43255000 mini=S11111111111111114phD2
[b47 t4] tid=12036 tIx=60180000 mini=S11111111111111116KSRD
[b46 t134] tid=11910 tIx=59550000 mini=S11111111111111116GD99
[b46 t142] tid=11918 tIx=59590000 mini=S11111111111111116GR2o
[b43 t245] tid=11253 tIx=56265000 mini=S11111111111111115yNdP
[b1 t215] tid=471 tIx=2355000 mini=S11111111111111111D54d
[b4 t100] tid=1124 tIx=5620000 mini=S11111111111111111Vodj
[b4 t118] tid=1142 tIx=5710000 mini=S11111111111111111WGPT
[b39 t107] tid=10091 tIx=50455000 mini=S11111111111111115TbWy
[b39 t19] tid=10003 tIx=50015000 mini=S11111111111111115RLim
[b40 t87] tid=10327 tIx=51635000 mini=S11111111111111115ZeHo
[b34 t167] tid=8871 tIx=44355000 mini=S11111111111111114vLCZ
[b26 t31] tid=6687 tIx=33435000 mini=S11111111111111113xN4h
[b41 t98] tid=10594 tIx=52970000 mini=S11111111111111115gV93
[b44 t161] tid=11425 tIx=57125000 mini=S111111111111111163nGz
[b24 t99] tid=6243 tIx=31215000 mini=S11111111111111113kz8r
[b9 t38] tid=2342 tIx=11710000 mini=S1111111111111111231yk
[b24 t43] tid=6187 tIx=30935000 mini=S11111111111111113jYuG
[b40 t197] tid=10437 tIx=52185000 mini=S11111111111111115cTna
[b27 t173] tid=7085 tIx=35425000 mini=S111111111111111148Zd4
[b5 t209] tid=1489 tIx=7445000 mini=S11111111111111111fA9G
[b48 t35] tid=12323 tIx=61615000 mini=S11111111111111116Snzb
[b48 t105] tid=12393 tIx=61965000 mini=S11111111111111116Ub35
[b48 t125] tid=12413 tIx=62065000 mini=S11111111111111116V6mD
[b19 t174] tid=5038 tIx=25190000 mini=S11111111111111113E77Y
[b10 t137] tid=2697 tIx=13485000 mini=S11111111111111112C7dC
[b49 t206] tid=12750 tIx=63750000 mini=S11111111111111116djew
[b4 t167] tid=1191 tIx=5955000 mini=S11111111111111111XXDd
[b39 t192] tid=10176 tIx=50880000 mini=S11111111111111115Vmrb
[b0 t207] tid=207 tIx=1035000 mini=S111111111111111116Jg3
[b0 t213] tid=213 tIx=1065000 mini=S111111111111111116TbH
[b0 t252] tid=252 tIx=1260000 mini=S111111111111111117TZM
[b35 t157] tid=9117 tIx=45585000 mini=S111111111111111152dqV
[b5 t89] tid=1369 tIx=6845000 mini=S11111111111111111c5nT
[b50 t181] tid=12981 tIx=64905000 mini=S11111111111111116jezj
[b51 t92] tid=13148 tIx=65740000 mini=S11111111111111116owDH
[b41 t5] tid=10501 tIx=52505000 mini=S11111111111111115e6uq
[b45 t221] tid=11741 tIx=58705000 mini=S11111111111111116BsxP
[b33 t123] tid=8571 tIx=42855000 mini=S11111111111111114neJX
[b22 t43] tid=5675 tIx=28375000 mini=S11111111111111113WRuM
[b44 t104] tid=11368 tIx=56840000 mini=S111111111111111162KZE
[b40 t32] tid=10272 tIx=51360000 mini=S11111111111111115YEYU
[b20 t60] tid=5180 tIx=25900000 mini=S11111111111111113HkAx
[b13 t38] tid=3366 tIx=16830000 mini=S11111111111111112VFye
[b36 t19] tid=9235 tIx=46175000 mini=S111111111111111155fDv
[b36 t158] tid=9374 tIx=46870000 mini=S111111111111111159Dpg
[b8 t142] tid=2190 tIx=10950000 mini=S11111111111111111y84L
[b11 t81] tid=2897 tIx=14485000 mini=S11111111111111112HEtc
[b38 t219] tid=9947 tIx=49735000 mini=S11111111111111115PuVE
[b14 t66] tid=3650 tIx=18250000 mini=S11111111111111112cY6Q
[b6 t194] tid=1730 tIx=8650000 mini=S11111111111111111mLMA
[b52 t75] tid=13387 tIx=66935000 mini=S11111111111111116v4Sj
[b4 t80] tid=1104 tIx=5520000 mini=S11111111111111111VHuf
[b8 t5] tid=2053 tIx=10265000 mini=S11111111111111111ucS1
[b12 t241] tid=3313 tIx=16565000 mini=S11111111111111112TuCh
[b7 t66] tid=1858 tIx=9290000 mini=S11111111111111111pcbf
[b55 t63] tid=14143 tIx=70715000 mini=S11111111111111117FS79
[b33 t187] tid=8635 tIx=43175000 mini=S11111111111111114pHRo
[b10 t195] tid=2755 tIx=13775000 mini=S11111111111111112DbqF
[b56 t226] tid=14562 tIx=72810000 mini=S11111111111111117SAsq
[b57 t250] tid=14842 tIx=74210000 mini=S11111111111111117ZM3m
[b56 t87] tid=14423 tIx=72115000 mini=S11111111111111117NcH5
[b35 t7] tid=8967 tIx=44835000 mini=S11111111111111114xntW
[b27 t126] tid=7038 tIx=35190000 mini=S111111111111111147MmQ
[b59 t103] tid=15207 tIx=76035000 mini=S11111111111111117ihZH
[b59 t180] tid=15284 tIx=76420000 mini=S11111111111111117kg1D
[b5 t121] tid=1401 tIx=7005000 mini=S11111111111111111cuM8
[b22 t102] tid=5734 tIx=28670000 mini=S11111111111111113Xwbc
[b22 t139] tid=5771 tIx=28855000 mini=S11111111111111113YtbG
[b10 t4] tid=2564 tIx=12820000 mini=S111111111111111128hwk
[b60 t17] tid=15377 tIx=76885000 mini=S11111111111111117o4ET
[b60 t104] tid=15464 tIx=77320000 mini=S11111111111111117qHYT
[b60 t111] tid=15471 tIx=77355000 mini=S11111111111111117qTwu
[b12 t72] tid=3144 tIx=15720000 mini=S11111111111111112Pa1m
[b61 t44] tid=15660 tIx=78300000 mini=S11111111111111117vJs1
[b28 t36] tid=7204 tIx=36020000 mini=S11111111111111114BcVm
[b5 t179] tid=1459 tIx=7295000 mini=S11111111111111111ePZ9
[b32 t191] tid=8383 tIx=41915000 mini=S11111111111111114hpsh
[b11 t172] tid=2988 tIx=14940000 mini=S11111111111111112Ka9V
[b7 t28] tid=1820 tIx=9100000 mini=S11111111111111111oe7q
[b62 t4] tid=15876 tIx=79380000 mini=S111111111111111181quh
[b45 t2] tid=11522 tIx=57610000 mini=S111111111111111166GT9
[b41 t33] tid=10529 tIx=52645000 mini=S11111111111111115epXh
[b6 t231] tid=1767 tIx=8835000 mini=S11111111111111111nHLs
[b18 t192] tid=4800 tIx=24000000 mini=S1111111111111111381NP
[b37 t179] tid=9651 tIx=48255000 mini=S11111111111111115GKY3
[b63 t115] tid=16243 tIx=81215000 mini=S11111111111111118BFPd
[b64 t99] tid=16483 tIx=82415000 mini=S11111111111111118HQ7H
[b64 t0] tid=16384 tIx=81920000 mini=S11111111111111118Erxo
[b64 t7] tid=16391 tIx=81955000 mini=S11111111111111118F3NF
[b64 t15] tid=16399 tIx=81995000 mini=S11111111111111118FFFu
[b35 t239] tid=9199 tIx=45995000 mini=S111111111111111154jiY
[b12 t159] tid=3231 tIx=16155000 mini=S11111111111111112RoKn
[b66 t73] tid=16969 tIx=84845000 mini=S11111111111111118VrTq
[b66 t34] tid=16930 tIx=84650000 mini=S11111111111111118UrVm
[b34 t75] tid=8779 tIx=43895000 mini=S11111111111111114syTe
[b13 t179] tid=3507 tIx=17535000 mini=S11111111111111112YsYt
[b67 t225] tid=17377 tIx=86885000 mini=S11111111111111118gJtF
[b67 t134] tid=17286 tIx=86430000 mini=S11111111111111118dydR
[b67 t127] tid=17279 tIx=86395000 mini=S11111111111111118doDy
[b68 t8] tid=17416 tIx=87080000 mini=S11111111111111118hJrK
[b21 t61] tid=5437 tIx=27185000 mini=S11111111111111113QLAD
[b28 t6] tid=7174 tIx=35870000 mini=S11111111111111114Aqub
[b69 t25] tid=17689 tIx=88445000 mini=S11111111111111118pJco
[b14 t3] tid=3587 tIx=17935000 mini=S11111111111111112avTT
[b14 t5] tid=3589 tIx=17945000 mini=S11111111111111112ayRs
[b25 t212] tid=6612 tIx=33060000 mini=S11111111111111113vSbK
[b71 t133] tid=18309 tIx=91545000 mini=S111111111111111196C95
[b71 t246] tid=18422 tIx=92110000 mini=S11111111111111119966T
[b0 t7] tid=7 tIx=35000 mini=S111111111111111111BQn
[b75 t27] tid=19227 tIx=96135000 mini=S11111111111111119Vib1
[b77 t185] tid=19897 tIx=99485000 mini=S11111111111111119ntRd
[b81 t148] tid=20884 tIx=104420000 mini=S1111111111111111AEBRq
[b27 t57] tid=6969 tIx=34845000 mini=S111111111111111145bDD
[b44 t88] tid=11352 tIx=56760000 mini=S111111111111111161un3
[b40 t140] tid=10380 tIx=51900000 mini=S11111111111111115b14w
[b82 t65] tid=21057 tIx=105285000 mini=S1111111111111111AJcZd
[b84 t229] tid=21733 tIx=108665000 mini=S1111111111111111AbwKV
[b84 t82] tid=21586 tIx=107930000 mini=S1111111111111111AYAq5
[b86 t15] tid=22031 tIx=110155000 mini=S1111111111111111AjaF9
[b85 t59] tid=21819 tIx=109095000 mini=S1111111111111111Ae99H
[b86 t100] tid=22116 tIx=110580000 mini=S1111111111111111Amkaj
[b87 t203] tid=22475 tIx=112375000 mini=S1111111111111111AvxB1
[b87 t226] tid=22498 tIx=112490000 mini=S1111111111111111AwYMm
[b25 t85] tid=6485 tIx=32425000 mini=S11111111111111113sBq5
[b88 t151] tid=22679 tIx=113395000 mini=S1111111111111111B2BPD
[b89 t141] tid=22925 tIx=114625000 mini=S1111111111111111B8V27
[b91 t199] tid=23495 tIx=117475000 mini=S1111111111111111BP6E3
[b1 t61] tid=317 tIx=1585000 mini=S1111111111111111198Ax
[b38 t68] tid=9796 tIx=48980000 mini=S11111111111111115L348
[b7 t109] tid=1901 tIx=9505000 mini=S11111111111111111qiWg
[b94 t97] tid=24161 tIx=120805000 mini=S1111111111111111BgA7q
[b95 t152] tid=24472 tIx=122360000 mini=S1111111111111111Bp8NB
[b95 t164] tid=24484 tIx=122420000 mini=S1111111111111111BpSCf
[b45 t119] tid=11639 tIx=58195000 mini=S111111111111111169GMS
[b6 t13] tid=1549 tIx=7745000 mini=S11111111111111111ghKr
[b96 t8] tid=24584 tIx=122920000 mini=S1111111111111111BrzqM
[b96 t59] tid=24635 tIx=123175000 mini=S1111111111111111BtJdu
[b96 t116] tid=24692 tIx=123460000 mini=S1111111111111111BumMh
[b98 t153] tid=25241 tIx=126205000 mini=S1111111111111111C9qMH
[b23 t137] tid=6025 tIx=30125000 mini=S11111111111111113fQ7w
[b39 t78] tid=10062 tIx=50310000 mini=S11111111111111115SrRB
[b5 t237] tid=1517 tIx=7585000 mini=S11111111111111111fsmF
[b17 t208] tid=4560 tIx=22800000 mini=S111111111111111131req
[b19 t124] tid=4988 tIx=24940000 mini=S11111111111111113CpoP
[b3 t249] tid=1017 tIx=5085000 mini=S11111111111111111T4bo
[b100 t24] tid=25624 tIx=128120000 mini=S1111111111111111CKecX
[b100 t238] tid=25838 tIx=129190000 mini=S1111111111111111CR8go
[b102 t92] tid=26204 tIx=131020000 mini=S1111111111111111CaWgX
[b102 t23] tid=26135 tIx=130675000 mini=S1111111111111111CYk8F
[b101 t192] tid=26048 tIx=130240000 mini=S1111111111111111CWWpF
[b104 t178] tid=26802 tIx=134010000 mini=S1111111111111111CqqWF
[b105 t86] tid=26966 tIx=134830000 mini=S1111111111111111Cv3GB
[b103 t254] tid=26622 tIx=133110000 mini=S1111111111111111CmDy1
[b105 t181] tid=27061 tIx=135305000 mini=S1111111111111111CxUTq
[b18 t154] tid=4762 tIx=23810000 mini=S1111111111111111372td
[b106 t53] tid=27189 tIx=135945000 mini=S1111111111111111D1kiK
[b106 t57] tid=27193 tIx=135965000 mini=S1111111111111111D1rf9
[b107 t86] tid=27478 tIx=137390000 mini=S1111111111111111D9AG7
[b39 t36] tid=10020 tIx=50100000 mini=S11111111111111115RmzW
[b108 t88] tid=27736 tIx=138680000 mini=S1111111111111111DFmjV
[b8 t189] tid=2237 tIx=11185000 mini=S11111111111111111zKvE
[b109 t49] tid=27953 tIx=139765000 mini=S1111111111111111DMLGP
[b11 t228] tid=3044 tIx=15220000 mini=S11111111111111112M1PC
[b11 t244] tid=3060 tIx=15300000 mini=S11111111111111112MRAW
[b113 t177] tid=29105 tIx=145525000 mini=S1111111111111111DrrWj
[b115 t29] tid=29469 tIx=147345000 mini=S1111111111111111E2BY3
[b115 t184] tid=29624 tIx=148120000 mini=S1111111111111111E69v7
[b116 t129] tid=29825 tIx=149125000 mini=S1111111111111111EBJfh
[b10 t235] tid=2795 tIx=13975000 mini=S11111111111111112EdHg
[b29 t54] tid=7478 tIx=37390000 mini=S11111111111111114Jdka
[b117 t185] tid=30137 tIx=150685000 mini=S1111111111111111EKJQF
[b118 t154] tid=30362 tIx=151810000 mini=S1111111111111111ER4po
[b38 t163] tid=9891 tIx=49455000 mini=S11111111111111115NUFq
[b121 t209] tid=31185 tIx=155925000 mini=S1111111111111111EnA55
[b121 t23] tid=30999 tIx=154995000 mini=S1111111111111111EhPcb
[b13 t218] tid=3546 tIx=17730000 mini=S11111111111111112ZsX5
[b119 t135] tid=30599 tIx=152995000 mini=S1111111111111111EX95q
[b120 t143] tid=30863 tIx=154315000 mini=S1111111111111111EduUT
[b120 t217] tid=30937 tIx=154685000 mini=S1111111111111111EfoTm
[b119 t5] tid=30469 tIx=152345000 mini=S1111111111111111ETorw
[b119 t187] tid=30651 tIx=153255000 mini=S1111111111111111EYUNb
[b122 t254] tid=31486 tIx=157430000 mini=S1111111111111111EusTM
[b128 t217] tid=32985 tIx=164925000 mini=S1111111111111111FaHTV
[b124 t215] tid=31959 tIx=159795000 mini=S1111111111111111F7zVD
[b124 t92] tid=31836 tIx=159180000 mini=S1111111111111111F4qfm
[b29 t10] tid=7434 tIx=37170000 mini=S11111111111111114HWMV
[b126 t49] tid=32305 tIx=161525000 mini=S1111111111111111FGrko
[b125 t180] tid=32180 tIx=160900000 mini=S1111111111111111FDexw
[b125 t188] tid=32188 tIx=160940000 mini=S1111111111111111FDrrb
[b125 t13] tid=32013 tIx=160065000 mini=S1111111111111111F9NkP
[b125 t224] tid=32224 tIx=161120000 mini=S1111111111111111FEnN3
[b130 t106] tid=33386 tIx=166930000 mini=S1111111111111111FkZUT
[b129 t198] tid=33222 tIx=166110000 mini=S1111111111111111FgMiX
[b129 t155] tid=33179 tIx=165895000 mini=S1111111111111111FfFod
[b129 t113] tid=33137 tIx=165685000 mini=S1111111111111111FeBNw
[b131 t227] tid=33763 tIx=168815000 mini=S1111111111111111FvDpT
[b131 t84] tid=33620 tIx=168100000 mini=S1111111111111111FrZGs
[b131 t222] tid=33758 tIx=168790000 mini=S1111111111111111Fv6PR
[b131 t31] tid=33567 tIx=167835000 mini=S1111111111111111FqCVu
[b132 t64] tid=33856 tIx=169280000 mini=S1111111111111111Fxc3h
[b134 t168] tid=34472 tIx=172360000 mini=S1111111111111111GEPd9
[b136 t173] tid=34989 tIx=174945000 mini=S1111111111111111GTe47
[b138 t176] tid=35504 tIx=177520000 mini=S1111111111111111GgqWf
[b136 t131] tid=34947 tIx=174735000 mini=S1111111111111111GSZdR
[b136 t149] tid=34965 tIx=174825000 mini=S1111111111111111GT2P9
[b137 t175] tid=35247 tIx=176235000 mini=S1111111111111111GaFXV
[b137 t183] tid=35255 tIx=176275000 mini=S1111111111111111GaTR9
[b137 t188] tid=35260 tIx=176300000 mini=S1111111111111111GaarB
[b140 t42] tid=35882 tIx=179410000 mini=S1111111111111111GrXLs
[b140 t50] tid=35890 tIx=179450000 mini=S1111111111111111GrjEX
[b18 t66] tid=4674 tIx=23370000 mini=S111111111111111134n6U
[b33 t74] tid=8522 tIx=42610000 mini=S11111111111111114mPUc
[b25 t127] tid=6527 tIx=32635000 mini=S11111111111111113tGFr
[b29 t202] tid=7626 tIx=38130000 mini=S11111111111111114NRjE
[b141 t113] tid=36209 tIx=181045000 mini=S1111111111111111GzuNX
[b141 t187] tid=36283 tIx=181415000 mini=S1111111111111111H2oMq
[b141 t34] tid=36130 tIx=180650000 mini=S1111111111111111GxsxB
[b141 t217] tid=36313 tIx=181565000 mini=S1111111111111111H3Zx3
[b142 t101] tid=36453 tIx=182265000 mini=S1111111111111111H7A31
[b143 t50] tid=36658 tIx=183290000 mini=S1111111111111111HCQjR
[b143 t24] tid=36632 tIx=183160000 mini=S1111111111111111HBk63
[b146 t106] tid=37482 tIx=187410000 mini=S1111111111111111HZXTu
[b26 t182] tid=6838 tIx=34190000 mini=S111111111111111142EWD
[b144 t83] tid=36947 tIx=184735000 mini=S1111111111111111HKpHD
[b144 t169] tid=37033 tIx=185165000 mini=S1111111111111111HN271
[b144 t179] tid=37043 tIx=185215000 mini=S1111111111111111HNGy5
[b144 t242] tid=37106 tIx=185530000 mini=S1111111111111111HPtc7
[b147 t33] tid=37665 tIx=188325000 mini=S1111111111111111HeDTm
[b145 t204] tid=37324 tIx=186620000 mini=S1111111111111111HVUdD
[b148 t7] tid=37895 tIx=189475000 mini=S1111111111111111Hk7KM
[b145 t12] tid=37132 tIx=185660000 mini=S1111111111111111HQZFV
[b148 t80] tid=37968 tIx=189840000 mini=S1111111111111111HmypT
[b37 t3] tid=9475 tIx=47375000 mini=S11111111111111115Bowo
[b155 t62] tid=39742 tIx=198710000 mini=S1111111111111111JZSZV
[b155 t233] tid=39913 tIx=199565000 mini=S1111111111111111Jdpis
[b155 t10] tid=39690 tIx=198450000 mini=S1111111111111111JY7Gj
[b3 t169] tid=937 tIx=4685000 mini=S11111111111111111R1hL
[b161 t166] tid=41382 tIx=206910000 mini=S1111111111111111KHU8o
[b159 t69] tid=40773 tIx=203865000 mini=S1111111111111111K1rxo
[b165 t238] tid=42478 tIx=212390000 mini=S1111111111111111KmZ9Z
[b166 t223] tid=42719 tIx=213595000 mini=S1111111111111111KsjMR
[b23 t39] tid=5927 tIx=29635000 mini=S11111111111111113ctTm
[b16 t103] tid=4199 tIx=20995000 mini=S11111111111111112rc6F
[b169 t85] tid=43349 tIx=216745000 mini=S1111111111111111L9sjm
[b170 t211] tid=43731 tIx=218655000 mini=S1111111111111111LKfWo
[b179 t203] tid=46027 tIx=230135000 mini=S1111111111111111MLW7q
[b171 t163] tid=43939 tIx=219695000 mini=S1111111111111111LQzfq
[b171 t168] tid=43944 tIx=219720000 mini=S1111111111111111LR86s
[b171 t26] tid=43802 tIx=219010000 mini=S1111111111111111LMV3V
[b180 t209] tid=46289 tIx=231445000 mini=S1111111111111111MTDY3
[b176 t17] tid=45073 tIx=225365000 mini=S1111111111111111Lv4AT
[b181 t176] tid=46512 tIx=232560000 mini=S1111111111111111MYvzB
[b176 t127] tid=45183 tIx=225915000 mini=S1111111111111111LxsfD
[b177 t63] tid=45375 tIx=226875000 mini=S1111111111111111M3o2w
[b181 t27] tid=46363 tIx=231815000 mini=S1111111111111111MV7XM
[b183 t230] tid=47078 tIx=235390000 mini=S1111111111111111MoSFH
[b184 t138] tid=47242 tIx=236210000 mini=S1111111111111111Mse1D
[b184 t118] tid=47222 tIx=236110000 mini=S1111111111111111Ms8H5
[b185 t162] tid=47522 tIx=237610000 mini=S1111111111111111MzpB9
[b188 t59] tid=48187 tIx=240935000 mini=S1111111111111111NHraj
[b189 t23] tid=48407 tIx=242035000 mini=S1111111111111111NPVaF
[b192 t130] tid=49282 tIx=246410000 mini=S1111111111111111Nmv7H
[b192 t201] tid=49353 tIx=246765000 mini=S1111111111111111Nojdy
[b194 t87] tid=49751 tIx=248755000 mini=S1111111111111111NywCK
[b197 t129] tid=50561 tIx=252805000 mini=S1111111111111111PLh7u
[b197 t125] tid=50557 tIx=252785000 mini=S1111111111111111PLbB5
[b199 t148] tid=51092 tIx=255460000 mini=S1111111111111111PaJMm
[b201 t27] tid=51483 tIx=257415000 mini=S1111111111111111PkKWf
[b201 t43] tid=51499 tIx=257495000 mini=S1111111111111111PkjHy
[b202 t32] tid=51744 tIx=258720000 mini=S1111111111111111Ps1Sf
[b203 t171] tid=52139 tIx=260695000 mini=S1111111111111111Q38YP
[b203 t253] tid=52221 tIx=261105000 mini=S1111111111111111Q5ERM
[b203 t217] tid=52185 tIx=260925000 mini=S1111111111111111Q4Juu
[b204 t87] tid=52311 tIx=261555000 mini=S1111111111111111Q7YBy
[b208 t158] tid=53406 tIx=267030000 mini=S1111111111111111QbbiX
[b214 t130] tid=54914 tIx=274570000 mini=S1111111111111111RGF6X
[b214 t243] tid=55027 tIx=275135000 mini=S1111111111111111RK93u
[b212 t179] tid=54451 tIx=272255000 mini=S1111111111111111R4Nvj
[b212 t136] tid=54408 tIx=272040000 mini=S1111111111111111R3H1q
[b215 t158] tid=55198 tIx=275990000 mini=S1111111111111111RPXDH
[b217 t3] tid=55555 tIx=277775000 mini=S1111111111111111RYfq9
[b25 t241] tid=6641 tIx=33205000 mini=S11111111111111113wBhX
[b25 t242] tid=6642 tIx=33210000 mini=S11111111111111113wDBj
[b218 t99] tid=55907 tIx=279535000 mini=S1111111111111111Rhh1y
[b218 t211] tid=56019 tIx=280095000 mini=S1111111111111111RkZV9
[b220 t118] tid=56438 tIx=282190000 mini=S1111111111111111RwJFq
[b221 t225] tid=56801 tIx=284005000 mini=S1111111111111111S6bnw
[b225 t231] tid=57831 tIx=289155000 mini=S1111111111111111SYzi3
[b226 t48] tid=57904 tIx=289520000 mini=S1111111111111111SasD9
[b232 t84] tid=59476 tIx=297380000 mini=S1111111111111111TH9iP
[b227 t119] tid=58231 tIx=291155000 mini=S1111111111111111SjFEo
[b226 t156] tid=58012 tIx=290060000 mini=S1111111111111111SddjV
[b230 t29] tid=58909 tIx=294545000 mini=S1111111111111111T2cy5
[b231 t116] tid=59252 tIx=296260000 mini=S1111111111111111TBQn3
[b231 t48] tid=59184 tIx=295920000 mini=S1111111111111111T9fhy
[b229 t8] tid=58632 tIx=293160000 mini=S1111111111111111SuXFm
[b229 t152] tid=58776 tIx=293880000 mini=S1111111111111111SyDHZ
[b1 t181] tid=437 tIx=2185000 mini=S11111111111111111CCXx
[b31 t55] tid=7991 tIx=39955000 mini=S11111111111111114XnEr
[b240 t251] tid=61691 tIx=308455000 mini=S1111111111111111UFuvf
[b239 t157] tid=61341 tIx=306705000 mini=S1111111111111111U6wiF
[b242 t181] tid=62133 tIx=310665000 mini=S1111111111111111UTEt7
[b243 t138] tid=62346 tIx=311730000 mini=S1111111111111111UYhUB
[b252 t116] tid=64628 tIx=323140000 mini=S1111111111111111VZBGK
[b245 t18] tid=62738 tIx=313690000 mini=S1111111111111111Uik7H
[b246 t244] tid=63220 tIx=316100000 mini=S1111111111111111Uw6X1
[b251 t141] tid=64397 tIx=321985000 mini=S1111111111111111VTFvX
[b17 t41] tid=4393 tIx=21965000 mini=S11111111111111112waST
[b249 t44] tid=63788 tIx=318940000 mini=S1111111111111111VBekX
[b248 t45] tid=63533 tIx=317665000 mini=S1111111111111111V57jm
[b250 t9] tid=64009 tIx=320045000 mini=S1111111111111111VHKEF
[b254 t100] tid=65124 tIx=325620000 mini=S1111111111111111VmtUw
[b255 t165] tid=65445 tIx=327225000 mini=S1111111111111111Vv7bM
[b254 t144] tid=65168 tIx=325840000 mini=S1111111111111111Vo1t3
[b254 t145] tid=65169 tIx=325845000 mini=S1111111111111111Vo3NF
[b256 t83] tid=65619 tIx=328095000 mini=S1111111111111111VzaDM
[b255 t34] tid=65314 tIx=326570000 mini=S1111111111111111VrktF
[b256 t136] tid=65672 tIx=328360000 mini=S1111111111111111W1vzK
[b260 t228] tid=66788 tIx=333940000 mini=S1111111111111111WWXjD
[b260 t125] tid=66685 tIx=333425000 mini=S1111111111111111WTtdu
[b258 t15] tid=66063 tIx=330315000 mini=S1111111111111111WBx9D
[b261 t53] tid=66869 tIx=334345000 mini=S1111111111111111WYc7y
[b261 t151] tid=66967 tIx=334835000 mini=S1111111111111111Wb7nF
[b257 t14] tid=65806 tIx=329030000 mini=S1111111111111111W5NA3
[b257 t75] tid=65867 tIx=329335000 mini=S1111111111111111W6vpf
[b259 t171] tid=66475 tIx=332375000 mini=S1111111111111111WNWWT
[b263 t214] tid=67542 tIx=337710000 mini=S1111111111111111WqrRD
[b263 t125] tid=67453 tIx=337265000 mini=S1111111111111111Woa8o
[b264 t110] tid=67694 tIx=338470000 mini=S1111111111111111WukLf
[b263 t243] tid=67571 tIx=337855000 mini=S1111111111111111WrbXD
[b265 t149] tid=67989 tIx=339945000 mini=S1111111111111111X3Joh
[b267 t104] tid=68456 tIx=342280000 mini=S1111111111111111XFGvK
[b267 t88] tid=68440 tIx=342200000 mini=S1111111111111111XEs91
[b268 t224] tid=68832 tIx=344160000 mini=S1111111111111111XQun7
[b268 t89] tid=68697 tIx=343485000 mini=S1111111111111111XMT8B
[b268 t136] tid=68744 tIx=343720000 mini=S1111111111111111XNeyu
[b268 t153] tid=68761 tIx=343805000 mini=S1111111111111111XP6FR
[b274 t55] tid=70199 tIx=350995000 mini=S1111111111111111Y1waw
[b275 t45] tid=70445 tIx=352225000 mini=S1111111111111111Y8FDq
[b275 t20] tid=70420 tIx=352100000 mini=S1111111111111111Y7c4f
[b282 t51] tid=72243 tIx=361215000 mini=S1111111111111111YvKdq
[b283 t228] tid=72676 tIx=363380000 mini=S1111111111111111Z7RDR
[b272 t164] tid=69796 tIx=348980000 mini=S1111111111111111XqcbZ
[b281 t60] tid=71996 tIx=359980000 mini=S1111111111111111YozWj
[b283 t54] tid=72502 tIx=362510000 mini=S1111111111111111Z2xbR
[b275 t164] tid=70564 tIx=352820000 mini=S1111111111111111YBJ6T
[b286 t32] tid=73248 tIx=366240000 mini=S1111111111111111ZN5Pm
[b287 t135] tid=73607 tIx=368035000 mini=S1111111111111111ZXGz3
[b287 t35] tid=73507 tIx=367535000 mini=S1111111111111111ZUiMM
[b286 t137] tid=73353 tIx=366765000 mini=S1111111111111111ZQmTV
[b288 t133] tid=73861 tIx=369305000 mini=S1111111111111111ZdnWb
[b292 t4] tid=74756 tIx=373780000 mini=S1111111111111111a2imm
[b293 t69] tid=75077 tIx=375385000 mini=S1111111111111111aAwtB
[b292 t114] tid=74866 tIx=374330000 mini=S1111111111111111a5YGX
[b289 t81] tid=74065 tIx=370325000 mini=S1111111111111111Zj1io
[b297 t197] tid=76229 tIx=381145000 mini=S1111111111111111agU8X
[b295 t230] tid=75750 tIx=378750000 mini=S1111111111111111aUCBR
[b299 t245] tid=76789 tIx=383945000 mini=S1111111111111111avpUP
[b299 t0] tid=76544 tIx=382720000 mini=S1111111111111111apYKh
[b299 t177] tid=76721 tIx=383605000 mini=S1111111111111111au5QK
[b294 t232] tid=75496 tIx=377480000 mini=S1111111111111111aMges
[b294 t45] tid=75309 tIx=376545000 mini=S1111111111111111aGtiB
[b294 t159] tid=75423 tIx=377115000 mini=S1111111111111111aKp9m
[b294 t11] tid=75275 tIx=376375000 mini=S1111111111111111aG2B9
[b294 t14] tid=75278 tIx=376390000 mini=S1111111111111111aG6dm
[b314 t2] tid=80386 tIx=401930000 mini=S1111111111111111cWznb
[b306 t101] tid=78437 tIx=392185000 mini=S1111111111111111bf3wM
[b303 t97] tid=77665 tIx=388325000 mini=S1111111111111111bKGVd
[b303 t20] tid=77588 tIx=387940000 mini=S1111111111111111bHJ3h
[b301 t233] tid=77289 tIx=386445000 mini=S1111111111111111b9ddq
[b309 t97] tid=79201 tIx=396005000 mini=S1111111111111111bzdVR
[b309 t95] tid=79199 tIx=395995000 mini=S1111111111111111bzaX1
[b310 t42] tid=79402 tIx=397010000 mini=S1111111111111111c5nF1
[b312 t78] tid=79950 tIx=399750000 mini=S1111111111111111cKpkP
[b310 t124] tid=79484 tIx=397420000 mini=S1111111111111111c7t7y
[b312 t249] tid=80121 tIx=400605000 mini=S1111111111111111cQCum
[b312 t127] tid=79999 tIx=399995000 mini=S1111111111111111cM5aX
[b315 t219] tid=80859 tIx=404295000 mini=S1111111111111111cj7pT
[b315 t122] tid=80762 tIx=403810000 mini=S1111111111111111cgdeP
[b316 t1] tid=80897 tIx=404485000 mini=S1111111111111111ck6JK
[b319 t97] tid=81761 tIx=408805000 mini=S1111111111111111d8EV5
[b322 t35] tid=82467 tIx=412335000 mini=S1111111111111111dSKq9
[b325 t142] tid=83342 tIx=416710000 mini=S1111111111111111dpkNB
[b324 t24] tid=82968 tIx=414840000 mini=S1111111111111111dfAUo
[b321 t190] tid=82366 tIx=411830000 mini=S1111111111111111dPjiF
[b322 t186] tid=82618 tIx=413090000 mini=S1111111111111111dWCGP
[b323 t86] tid=82774 tIx=413870000 mini=S1111111111111111daC8f
[b323 t196] tid=82884 tIx=414420000 mini=S1111111111111111dd1dR
[b323 t209] tid=82897 tIx=414485000 mini=S1111111111111111ddLx7
[b326 t32] tid=83488 tIx=417440000 mini=S1111111111111111dtVNP
[b335 t40] tid=85800 tIx=429000000 mini=S1111111111111111eujkj
[b333 t151] tid=85399 tIx=426995000 mini=S1111111111111111ejTjm
[b330 t78] tid=84558 tIx=422790000 mini=S1111111111111111eMujm
[b327 t176] tid=83888 tIx=419440000 mini=S1111111111111111e4ju9
[b330 t12] tid=84492 tIx=422460000 mini=S1111111111111111eLDe7
[b339 t52] tid=86836 tIx=434180000 mini=S1111111111111111fNHb5
[b340 t182] tid=87222 tIx=436110000 mini=S1111111111111111fYBJw
[b327 t96] tid=83808 tIx=419040000 mini=S1111111111111111e2gzb
[b332 t26] tid=85018 tIx=425090000 mini=S1111111111111111eZhSw
[b335 t138] tid=85898 tIx=429490000 mini=S1111111111111111exFR1
[b335 t92] tid=85852 tIx=429260000 mini=S1111111111111111ew53V
[b338 t138] tid=86666 tIx=433330000 mini=S1111111111111111fHvuu
[b340 t90] tid=87130 tIx=435650000 mini=S1111111111111111fVpZu
[b338 t38] tid=86566 tIx=432830000 mini=S1111111111111111fFNHD
[b338 t241] tid=86769 tIx=433845000 mini=S1111111111111111fLa1D
[b337 t174] tid=86446 tIx=432230000 mini=S1111111111111111fCHvP
[b343 t171] tid=87979 tIx=439895000 mini=S1111111111111111fsaTZ
[b337 t149] tid=86421 tIx=432105000 mini=S1111111111111111fBemD
[b327 t252] tid=83964 tIx=419820000 mini=S1111111111111111e6grs
[b328 t5] tid=83973 tIx=419865000 mini=S1111111111111111e6vEj
[b344 t28] tid=88092 tIx=440460000 mini=S1111111111111111fvUQw
[b347 t14] tid=88846 tIx=444230000 mini=S1111111111111111gFo6w
[b345 t117] tid=88437 tIx=442185000 mini=S1111111111111111g5KCK
[b345 t68] tid=88388 tIx=441940000 mini=S1111111111111111g44NB
[b348 t172] tid=89260 tIx=446300000 mini=S1111111111111111gSQSb
[b349 t98] tid=89442 tIx=447210000 mini=S1111111111111111gX4xF
[b349 t63] tid=89407 tIx=447035000 mini=S1111111111111111gWAw1
[b351 t226] tid=90082 tIx=450410000 mini=S1111111111111111goUCf
[b351 t113] tid=89969 tIx=449845000 mini=S1111111111111111gkaFH
[b352 t185] tid=90297 tIx=451485000 mini=S1111111111111111gtym9
[b352 t22] tid=90134 tIx=450670000 mini=S1111111111111111gpoVR
[b352 t95] tid=90207 tIx=451035000 mini=S1111111111111111grfzX
[b354 t157] tid=90781 tIx=453905000 mini=S1111111111111111h7P9H
[b361 t96] tid=92512 tIx=462560000 mini=S1111111111111111hsjyR
[b361 t106] tid=92522 tIx=462610000 mini=S1111111111111111hszqV
[b353 t57] tid=90425 tIx=452125000 mini=S1111111111111111gxG1d
[b357 t156] tid=91548 tIx=457740000 mini=S1111111111111111hT39y
[b365 t230] tid=93670 tIx=468350000 mini=S1111111111111111iPR91
[b356 t178] tid=91314 tIx=456570000 mini=S1111111111111111hM3MZ
[b364 t180] tid=93364 tIx=466820000 mini=S1111111111111111iFaKh
[b356 t243] tid=91379 tIx=456895000 mini=S1111111111111111hNhy1
[b358 t91] tid=91739 tIx=458695000 mini=S1111111111111111hXw3V
[b364 t124] tid=93308 tIx=466540000 mini=S1111111111111111iE967
[b365 t54] tid=93494 tIx=467470000 mini=S1111111111111111iJuYb
[b360 t142] tid=92302 tIx=461510000 mini=S1111111111111111hnMqy
[b360 t185] tid=92345 tIx=461725000 mini=S1111111111111111hoTks
[b359 t24] tid=91928 tIx=459640000 mini=S1111111111111111hcmxb
[b359 t240] tid=92144 tIx=460720000 mini=S1111111111111111hiK1H
[b365 t205] tid=93645 tIx=468225000 mini=S1111111111111111iNmyq
[b367 t181] tid=94133 tIx=470665000 mini=S1111111111111111ibHJo
[b367 t193] tid=94145 tIx=470725000 mini=S1111111111111111ibb9H

Work finished at Sun Sep 14 19:55:49 2025
------------------------
source: https://github.com/PawelGorny/WifSolverCuda